# NammaMart Quick Commerce — Part 0: Exploratory Data Profiling
**Student / Engineer**: Shubham Sharma (`shubham@cossth.com`)  
**Module**: IIT Roorkee CEC & Futurense · Data Pipeline to ML Model (FDAIE Module 2)  

This exploratory notebook inspects all 5 raw source datasets and the late-arriving increment file for the NammaMart platform:
1. `orders.csv`
2. `payments.csv`
3. `customers.csv`
4. `products.csv`
5. `stores.csv`
6. `orders_2026-10-01_increment.csv`

It profiles null distributions, detects schema and referential integrity violations, catalogs all 17 data quality anomalies, defines candidate validation rules, and formulates client clarification questions prior to building the Airflow ETL and ELT pipelines.

In [1]:
import os
import pandas as pd
import numpy as np
import duckdb

# Configure data paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR = os.path.join(BASE_DIR, 'include', 'data', 'nammamart')
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.join(os.getcwd(), 'include', 'data', 'nammamart')

print(f"Data Directory: {DATA_DIR}")
files = {
    'orders': os.path.join(DATA_DIR, 'orders.csv'),
    'payments': os.path.join(DATA_DIR, 'payments.csv'),
    'customers': os.path.join(DATA_DIR, 'customers.csv'),
    'products': os.path.join(DATA_DIR, 'products.csv'),
    'stores': os.path.join(DATA_DIR, 'stores.csv'),
    'increment': os.path.join(DATA_DIR, 'orders_2026-10-01_increment.csv')
}

dfs = {name: pd.read_csv(p) for name, p in files.items() if os.path.exists(p)}
for name, df in dfs.items():
    print(f"{name:<12}: {len(df):>5} rows | {len(df.columns):>2} columns")

## 1. Schema & Summary Inspection

In [2]:
for name, df in dfs.items():
    print(f"\n{'='*30} {name.upper()} {'='*30}")
    display(df.head(2))
    print("\nNull counts:")
    nulls = df.isnull().sum()
    print(nulls[nulls > 0] if nulls.sum() > 0 else "No nulls found.")

## 2. Deep Dive: 17 Data Quality Issues Cataloging

We systematically check each quality dimension (Completeness, Uniqueness, Validity, Accuracy, Consistency, Integrity):

In [3]:
issues_found = []

# Check 1: Orders with negative or zero quantity
df_ord = dfs['orders']
bad_qty = df_ord[df_ord['quantity'] <= 0]
issues_found.append(('orders', 'quantity', 'Validity', f'{len(bad_qty)} rows with quantity <= 0', 'Critical'))

# Check 2: Delivery minutes out of SLA (<8 or >120 for delivered)
bad_del = df_ord[(df_ord['order_status'].str.upper() == 'DELIVERED') & ((df_ord['delivery_minutes'] < 8) | (df_ord['delivery_minutes'] > 120) | (df_ord['delivery_minutes'].isnull()))]
issues_found.append(('orders', 'delivery_minutes', 'Validity/Accuracy', f'{len(bad_del)} rows delivered out of realistic SLA [8, 120]', 'Critical'))

# Check 3: Calculation formula mismatch (order_amount != quantity * unit_price * (1 - discount/100))
calc_amt = (df_ord['quantity'] * df_ord['unit_price'] * (1 - df_ord['discount_pct'] / 100.0)).round(2)
bad_amt = df_ord[abs(df_ord['order_amount'] - calc_amt) > 0.05]
issues_found.append(('orders', 'order_amount', 'Accuracy', f'{len(bad_amt)} rows with formula calculation discrepancy', 'Critical'))

# Check 4: Unreconciled payment orders
df_pay = dfs['payments']
paid_orders = set(df_pay[df_pay['payment_status'].str.upper() == 'SUCCESS']['order_id'])
del_unpaid = df_ord[(df_ord['order_status'].str.upper() == 'DELIVERED') & (~df_ord['order_id'].isin(paid_orders))]
issues_found.append(('orders/payments', 'order_id', 'Consistency', f'{len(del_unpaid)} delivered orders without successful payment match', 'Critical'))

# Check 5: Duplicate customer phones
df_cust = dfs['customers']
dup_phones = df_cust[df_cust['phone'].duplicated(keep=False)]
issues_found.append(('customers', 'phone', 'Uniqueness', f'{len(dup_phones)} customers sharing duplicate phone numbers', 'Warning'))

# Check 6: Unmasked customer PII
issues_found.append(('customers', 'email,phone', 'Compliance', 'Raw PII stored in plain text requiring DPDP Act 2023 masking', 'Critical'))

# Check 7: Product cost_price > mrp
df_prod = dfs['products']
bad_cost = df_prod[df_prod['cost_price'] > df_prod['mrp']]
issues_found.append(('products', 'cost_price', 'Accuracy', f'{len(bad_cost)} products where cost_price exceeds MRP (negative margin)', 'Critical'))

# Check 8: Missing store zones
df_store = dfs['stores']
null_zones = df_store[df_store['zone'].isnull() | (df_store['zone'] == '')]
issues_found.append(('stores', 'zone', 'Completeness', f'{len(null_zones)} dark stores missing zone assignment', 'Warning'))

issues_df = pd.DataFrame(issues_found, columns=['Entity', 'Field', 'Dimension', 'Description', 'Severity'])
display(issues_df)

## 3. Client Clarification Questions (Part 0 Deliverable)

1. **Returned / Cancelled Delivered Orders**: For orders marked as `RETURNED` with payments captured (`SUCCESS`), should revenue be recognized initially and offset by a refund mart entry, or excluded entirely from delivered net revenue?  
   *Recommended Action*: Model gross orders in staging, exclude refunds from net delivered sales.
2. **Orphan Customer IDs in Orders**: Several historical orders reference `customer_id` values not present in `customers.csv`.  
   *Recommended Action*: Quarantine orders to `include/quarantine/` during ETL; map to a surrogate synthetic customer (`UNKNOWN_CUSTOMER_999`) in ELT.
3. **Late-Arriving Increment Updates**: The `orders_2026-10-01_increment.csv` file contains both net-new orders and status updates for previous September orders.  
   *Recommended Action*: Use an idempotent `MERGE INTO stg_orders` on `order_id` updating status and amounts while preserving original `created_at`.

## 4. Candidate Validation Rules

| Rule ID | Entity | Field | Rule Type | Condition / Expression | Target Action |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **VR-01** | `orders` | `quantity` | Range Check | `quantity > 0` | Quarantine / Reject |
| **VR-02** | `orders` | `delivery_minutes` | Range Check | `status != 'DELIVERED' OR (delivery_minutes BETWEEN 8 AND 120)` | Quarantine |
| **VR-03** | `orders` | `order_amount` | Formula Consistency | `abs(order_amount - qty * price * (1 - disc/100)) <= 0.05` | Quarantine |
| **VR-04** | `customers` | `email` | Regex Validation | `email ~ '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'` | Mask & Quarantine |
| **VR-05** | `products` | `cost_price` | Cross-Column Logic | `cost_price <= mrp` | Flag / Review |
| **VR-06** | `payments` | `payment_status` | Set Membership | `payment_status IN ('SUCCESS', 'FAILED', 'PENDING', 'REFUNDED')` | Flag / Reject |

In [4]:
# Export Candidate Issues to CSV
output_csv = os.path.join(BASE_DIR, 'include', 'data_issues_log.csv')
issues_df.to_csv(output_csv, index=False)
print(f"Data profiling complete. Candidate issues exported to {output_csv}")